In [ ]:
import humanoid_bench
import gymnasium as gym
from fast_td3.environments.humanoid_bench_env import HumanoidBenchEnv

# H1

In [ ]:
env = gym.make(
        "h1-maze-v0",
        render_mode="rgb_array",
)
env.reset()
env.step(env.action_space.sample())
data = env.unwrapped.named.data

In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation as R

# Define body tree structure and joint order
# Only include main bodies for simplicity; you can extend to every link
body_tree = {
    "pelvis": {
        "pos": np.array([0,0,0]), "quat": None, "joint": "free_base", "children": [
            {"name": "left_hip_yaw_link", "pos": np.array([0, 0.0875, -0.1742]), "joint": "left_hip_yaw", "children": [
                {"name": "left_hip_roll_link", "pos": np.array([0.039468, 0, 0]), "joint": "left_hip_roll", "children": [
                    {"name": "left_hip_pitch_link", "pos": np.array([0,0.11536,0]), "joint": "left_hip_pitch", "children": [
                        {"name": "left_knee_link", "pos": np.array([0,0,-0.4]), "joint": "left_knee", "children": [
                            {"name": "left_ankle_link", "pos": np.array([0,0,-0.4]), "joint": "left_ankle", "children": []}
                        ]}
                    ]}
                ]}
            ]},
            {"name": "right_hip_yaw_link", "pos": np.array([0, -0.0875, -0.1742]), "joint": "right_hip_yaw", "children": [
                {"name": "right_hip_roll_link", "pos": np.array([0.039468,0,0]), "joint": "right_hip_roll", "children": [
                    {"name": "right_hip_pitch_link", "pos": np.array([0,-0.11536,0]), "joint": "right_hip_pitch", "children": [
                        {"name": "right_knee_link", "pos": np.array([0,0,-0.4]), "joint": "right_knee", "children": [
                            {"name": "right_ankle_link", "pos": np.array([0,0,-0.4]), "joint": "right_ankle", "children": []}
                        ]}
                    ]}
                ]}
            ]},
            {"name": "torso_link", "pos": np.array([0,0,0]), "joint": "torso", "children": [
                {"name": "left_shoulder_pitch_link", "pos": np.array([0.0055,0.15535,0.42999]), "joint": "left_shoulder_pitch", "children": [
                    {"name": "left_shoulder_roll_link", "pos": np.array([-0.0055,0.0565,-0.0165]), "joint": "left_shoulder_roll", "children": [
                        {"name": "left_shoulder_yaw_link", "pos": np.array([0,0,-0.1343]), "joint": "left_shoulder_yaw", "children": [
                            {"name": "left_elbow_link", "pos": np.array([0.0185,0,-0.198]), "joint": "left_elbow", "children": []}
                        ]}
                    ]}
                ]},
                {"name": "right_shoulder_pitch_link", "pos": np.array([0.0055,-0.15535,0.42999]), "joint": "right_shoulder_pitch", "children": [
                    {"name": "right_shoulder_roll_link", "pos": np.array([-0.0055,-0.0565,-0.0165]), "joint": "right_shoulder_roll", "children": [
                        {"name": "right_shoulder_yaw_link", "pos": np.array([0,0,-0.1343]), "joint": "right_shoulder_yaw", "children": [
                            {"name": "right_elbow_link", "pos": np.array([0.0185,0,-0.198]), "joint": "right_elbow", "children": []}
                        ]}
                    ]}
                ]}
            ]}
        ]
    }
}

# Joint order in qpos (matching your FieldIndexer)
joint_indices = {
    "free_base": slice(0,7),   # 0-6: 3 pos + 4 quat
    "left_hip_yaw": 7,
    "left_hip_roll": 8,
    "left_hip_pitch": 9,
    "left_knee": 10,
    "left_ankle": 11,
    "right_hip_yaw": 12,
    "right_hip_roll": 13,
    "right_hip_pitch": 14,
    "right_knee": 15,
    "right_ankle": 16,
    "torso": 17,
    "left_shoulder_pitch": 18,
    "left_shoulder_roll": 19,
    "left_shoulder_yaw": 20,
    "left_elbow": 21,
    "right_shoulder_pitch": 22,
    "right_shoulder_roll": 23,
    "right_shoulder_yaw": 24,
    "right_elbow": 25
}

# Joint axes (from XML)
joint_axes = {
    "left_hip_yaw": np.array([0,0,1]),
    "left_hip_roll": np.array([1,0,0]),
    "left_hip_pitch": np.array([0,1,0]),
    "left_knee": np.array([0,1,0]),
    "left_ankle": np.array([0,1,0]),
    "right_hip_yaw": np.array([0,0,1]),
    "right_hip_roll": np.array([1,0,0]),
    "right_hip_pitch": np.array([0,1,0]),
    "right_knee": np.array([0,1,0]),
    "right_ankle": np.array([0,1,0]),
    "torso": np.array([0,0,1]),
    "left_shoulder_pitch": np.array([0,1,0]),
    "left_shoulder_roll": np.array([1,0,0]),
    "left_shoulder_yaw": np.array([0,0,1]),
    "left_elbow": np.array([0,1,0]),
    "right_shoulder_pitch": np.array([0,1,0]),
    "right_shoulder_roll": np.array([1,0,0]),
    "right_shoulder_yaw": np.array([0,0,1]),
    "right_elbow": np.array([0,1,0])
}

def fk_joint_positions(body, qpos, parent_T=np.eye(4)):
    """
    Compute world-space joint anchor positions for the simplified tree.
    - Applies free base pose first.
    - Computes anchor = parent_T @ pos_offset (rotated) and records it.
    - Applies joint rotation about the anchor (no translation).
    - Recurse to children using the updated transform (origin moved to anchor).
    """
    # Start from parent transform
    T = parent_T.copy()

    # Identify this joint
    joint_name = body.get("joint", body.get("name", f"unnamed_{id(body)}"))
    pos_offset = body.get("pos", np.zeros(3))

    # If free base, set base pose first (pos + orientation)
    if joint_name == "free_base":
        base = qpos[joint_indices[joint_name]]
        base_pos = base[:3]
        base_quat_wxyz = base[3:7]  # [w,x,y,z]
        # scipy expects [x,y,z,w]
        base_quat_xyzw = [base_quat_wxyz[1], base_quat_wxyz[2], base_quat_wxyz[3], base_quat_wxyz[0]]
        T[:3, :3] = R.from_quat(base_quat_xyzw).as_matrix()
        T[:3, 3] = base_pos

    # Anchor world position: translate by offset expressed in parent's rotated frame
    anchor_world = T[:3, 3] + T[:3, :3] @ pos_offset

    # Record anchor position for this joint (strip optional _link suffix)
    joint_key = joint_name.replace("_link", "")
    joint_pos = {joint_key: anchor_world}

    # Rotate about anchor if revolute joint
    if joint_name in joint_axes:
        axis_local = joint_axes[joint_name]
        angle = qpos[joint_indices[joint_name]]
        R_joint = R.from_rotvec(axis_local * angle).as_matrix()
        T[:3, :3] = T[:3, :3] @ R_joint

    # Move origin to the anchor for children
    T[:3, 3] = anchor_world

    # Recurse
    for child in body.get("children", []):
        joint_pos.update(fk_joint_positions(child, qpos, T))

    return joint_pos


joint_positions = fk_joint_positions(body_tree["pelvis"], data.qpos)

for name, pos in joint_positions.items():
    print(f"{name}: {pos}")


In [ ]:
print(data.xanchor)

In [ ]:
# Build mapping from joint order to xanchor indices (assumed order)
joint_to_xanchor_mapping = {
    "free_base": 0,
    "left_hip_yaw": 1,
    "left_hip_roll": 2,
    "left_hip_pitch": 3,
    "left_knee": 4,
    "left_ankle": 5,
    "right_hip_yaw": 6,
    "right_hip_roll": 7,
    "right_hip_pitch": 8,
    "right_knee": 9,
    "right_ankle": 10,
    "torso": 11,
    "left_shoulder_pitch": 12,
    "left_shoulder_roll": 13,
    "left_shoulder_yaw": 14,
    "left_elbow": 15,
    "right_shoulder_pitch": 16,
    "right_shoulder_roll": 17,
    "right_shoulder_yaw": 18,
    "right_elbow": 19,
}

print(f"{'Joint':20s} | {'FK':30s} | {'Xanchor':30s} | {'Max Diff':8s}")
print("-" * 100)

max_diff_overall = 0.0
max_diff_joint = ""
sum_err = 0.0
count = 0

for joint_name, xanchor_idx in joint_to_xanchor_mapping.items():
    if joint_name in joint_positions:
        fk_pos = joint_positions[joint_name]
        xanchor_pos = data.xanchor[xanchor_idx]
        diff = np.abs(fk_pos - xanchor_pos)
        max_diff = np.max(diff)
        rmse = np.sqrt(np.mean((fk_pos - xanchor_pos) ** 2))
        
        sum_err += rmse
        count += 1
        
        if max_diff > max_diff_overall:
            max_diff_overall = max_diff
            max_diff_joint = joint_name
        
        fk_str = np.array2string(fk_pos, precision=3, separator=',', suppress_small=True)
        xanchor_str = np.array2string(xanchor_pos, precision=3, separator=',', suppress_small=True)
        
        print(f"{joint_name:20s} | {fk_str:30s} | {xanchor_str:30s} | {max_diff:8.6f}")

avg_rmse = (sum_err / max(count,1))
print(f"\nLargest difference: {max_diff_overall:.6f} at joint '{max_diff_joint}'")
print(f"Average RMSE: {avg_rmse:.6f}")


# G1

In [ ]:
env = gym.make(
        "g1hand-stand-v0",
        render_mode="rgb_array",
)
env.reset()
data = env.unwrapped.named.data
print(data.qpos)
print(data.qvel)
print(data.xanchor)

In [ ]:
env = gym.make(
        "g1hand-stand-v0",
        render_mode="rgb_array",
)
env.reset()
data = env.unwrapped.named.data
print(data.qpos)
print(data.xanchor)